# Trabalho de EST2 - G13

Fazer as regressões Ridge, Lasso e ElasticNet com a variável dependente “lwage” (salário-hora da esposa em logaritmo neperiano) e todas as demais variáveis da base de dados são variáveis explicativas (todas essas variáveis tentam explicar o salário-hora da esposa). No pdf você deve colocar a rotina utilizada, mostrar em uma tabela as estatísticas dos modelos (RMSE e R2) e concluir qual o melhor modelo entre os três, e mostrar o resultado da predição com intervalos de confiança para os seguintes valores:  
  
husage = 40 (anos – idade do marido)  
husunion = 0 (marido não possui união estável)  
husearns = 600 (US$ renda do marido por semana)  
huseduc = 13 (anos de estudo do marido)  
husblck = 1 (o marido é preto)  
hushisp = 0 (o marido não é hispânico)  
hushrs = 40 (horas semanais de trabalho do marido)  
kidge6 = 1 (possui filhos maiores de 6 anos)  
earns = xxx   
age = 38 (anos – idade da esposa)  
black = 0 (a esposa não é preta)  
educ = 13 (anos de estudo da esposa)  
hispanic = 1 (a esposa é hispânica)  
union = 0 (esposa não possui união estável)  
exper = 18 (anos de experiência de trabalho da esposa)  
kidlt6 = 1 (possui filhos menores de 6 anos)  

obs: lembre-se de que a variável dependente “lwage” já está em logarítmo, portanto voçê não precisa aplicar o logaritmo nela para fazer as regressões, mas é necessário aplicar o antilog para obter o resultado da predição.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import  r2_score, root_mean_squared_error

# Para medir o tempo do processo
import time

In [2]:
# Vamos utilizar a base de dados "trabalhosalarios" 
# Carregando o dataset
df = pd.read_csv('trabalhosalarios.csv')

In [3]:
df.head()

,husage,husunion,husearns,huseduc,husblck,hushisp,hushrs,kidge6,earns,age,black,educ,hispanic,union,exper,kidlt6,lwage
0,56,0,1500,14,0,0,40,1,100.0,49,0,12,0,0,31,0,1.897120
1,31,0,800,17,0,0,40,0,480.0,29,0,14,0,0,9,0,2.484907
2,33,0,950,13,0,0,60,0,455.0,30,0,12,0,0,12,1,2.431418
3,34,0,1000,14,0,0,50,1,102.0,31,0,12,0,0,13,0,1.629241
4,42,0,730,14,0,0,40,1,300.0,41,0,12,0,0,23,0,2.302585


In [4]:
y = df[['lwage']].to_numpy() 
y = y.ravel() # garantir o shape devido a regressão Lasso aguardar y com o shape (n,) ao inves de (n,1)
X = df.drop(['lwage'], axis=1)

In [5]:
# Vamos particionar o dataset em 80% para treinamento e 20% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
# Vamos checar as dimensoes das bases de treinamento e teste
print(f"Total de linhas e colunas do dataset : {df.shape} ")
print(f"Total de linhas e colunas do Treino : {X_treino.shape}")
print(f"Total de linhas e colunas do Teste : {X_teste.shape}")

Total de linhas e colunas do dataset : (2574, 17) 
Total de linhas e colunas do Treino : (2059, 16)
Total de linhas e colunas do Teste : (515, 16)


In [7]:
# A base de treinamento possui 2059 linhas (observações) e  16 colunas (variaveis)
# A base de teste possui 515 linhas (observações) e 16 colunas (variaveis)

In [8]:
print(df.columns)

Index(['husage', 'husunion', 'husearns', 'huseduc', 'husblck', 'hushisp',
       'hushrs', 'kidge6', 'earns', 'age', 'black', 'educ', 'hispanic',
       'union', 'exper', 'kidlt6', 'lwage'],
      dtype='str')


In [9]:
# Vamos listar as colunas que necessitam padronizar (contínuas) e as que não devem ser padronizadas (binárias)
col_continuas = ['husage', 'husearns', 'huseduc', 'hushrs', 'earns', 'age', 'educ', 'exper']
col_binarias = ['husunion','husblck', 'hushisp', 'kidge6', 'black', 'hispanic', 'union', 'kidlt6']

In [10]:
# Criar um preprocessamento para transformar apenas as variáveis contínuas e passar as binárias
processamento_X = ColumnTransformer(
    transformers=[
        ("cont", StandardScaler(), col_continuas),
        ("bin", "passthrough", col_binarias)
    ]
)


In [11]:
#############################################################
#                     Regressões                            #
#############################################################

In [12]:
alphas = np.logspace(-4, 4, 100)

models = {
    "Ridge": RidgeCV(alphas=alphas, cv=5),
    "Lasso": LassoCV(alphas=alphas, cv=5),
    "ElasticNet": ElasticNetCV(alphas=alphas, cv=5)
}

In [13]:
trained_models = {}

for name, model_reg in models.items():
    start_time = time.time() # inicia contagem de tempo

    # Busca o melhor Alpha e avalia pelo R²
    pipeline = Pipeline(steps=[
        ("preprocessamento", processamento_X),
        ("modelo", model_reg)
    ])
    
    # Wrapper para transformar y
    model = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=StandardScaler()
    )
    
    model.fit(X_treino, y_treino) # treina

    y_hat = model.predict(X_teste) # faz a predição

    train_time = time.time() - start_time # calcula o tempo do processo

    trained_models[name] = {
        "model": model,
        "alpha": model.regressor_.named_steps['modelo'].alpha_,
        "coeficientes": model.regressor_.named_steps['modelo'].coef_,
        "predict": y_hat,
        "train_time": train_time
    } # salva no dicionario

    print(f'Tempo total de treinamento e predição {name} : {train_time:.2f} segundos')

Tempo total de treinamento e predição Ridge : 0.81 segundos
Tempo total de treinamento e predição Lasso : 0.08 segundos
Tempo total de treinamento e predição ElasticNet : 0.05 segundos


In [14]:
# obter o RMSEe R² para cada modelo
def metricas(y_true, y_hat, tempo, alfa):
    """Recebe os valores reais, predito e o tempo do modelo e retorna um dicionário com
    o RMSE e R² e tempo de duração"""

    t = round(tempo,2)
    rmse_calc = round(root_mean_squared_error(y_true, y_hat),2)
    r2_calc = round(r2_score(y_true, y_hat) * 100,2)
    metrica = {"rmse": rmse_calc, "r2": r2_calc, 'tempo': t, 'alpha': alfa}
    return metrica

In [15]:
# cria um dataframe para guardar as medidas
df_met = pd.DataFrame(columns=['rmse','r2', 'tempo', 'alpha'])

# insere no dataframe as metricas calculadas de cada modelo
for name, info in trained_models.items():
    y_hat = info['predict']
    t = info['train_time']
    alfa = info['alpha']
    df_met.loc[name] = metricas(y_teste,y_hat, t, alfa)


print(df_met)

            rmse     r2  tempo      alpha
Ridge       0.26  72.96   0.81  31.257158
Lasso       0.26  72.85   0.08   0.004977
ElasticNet  0.26  72.86   0.05   0.008697


In [17]:
# Lista os coeficientes

df_coef = pd.DataFrame({
    "Variavel": X_treino.columns
}).set_index("Variavel")

for name, info in trained_models.items():
    df_coef[name] = pd.Series(
        info["coeficientes"],
        index= X_treino.columns
    )
    
print(df_coef)


             Ridge     Lasso  ElasticNet
Variavel                                
husage    0.016018  0.012665    0.012650
husunion  0.076943  0.072830    0.073699
husearns  0.021374  0.021029    0.021430
huseduc  -0.023133 -0.016641   -0.017466
husblck   0.719394  0.729443    0.725975
hushisp   0.016935  0.000138    0.001168
hushrs    0.104622  0.107155    0.107615
kidge6   -0.011880  0.000000    0.000000
earns     0.021206  0.002081    0.004545
age       0.031876 -0.000000   -0.000000
black    -0.055843 -0.002816   -0.012868
educ      0.013104  0.000000    0.000000
hispanic -0.073822 -0.000000   -0.000000
union    -0.040338 -0.000000   -0.000686
exper     0.094275  0.064788    0.068404
kidlt6    0.119945  0.083476    0.086034


In [18]:
# Vamos criar uma nova observação para prever o valor da hora salario da esposa

# husage = 40 (anos – idade do marido)  
# husunion = 0 (marido não possui união estável)  
# husearns = 600 (US$ renda do marido por semana)  
# huseduc = 13 (anos de estudo do marido)  
# husblck = 1 (o marido é preto)  
# hushisp = 0 (o marido não é hispânico)  
# hushrs = 40 (horas semanais de trabalho do marido)  
# kidge6 = 1 (possui filhos maiores de 6 anos)
# earns = 355.5 (rendimento da esposa em US$)  
# age = 38 (anos – idade da esposa)  
# black = 0 (a esposa não é preta)  
# educ = 13 (anos de estudo da esposa)  
# hispanic = 1 (a esposa é hispânica)  
# union = 0 (esposa não possui união estável)  
# exper = 18 (anos de experiência de trabalho da esposa)  
# kidlt6 = 1 (possui filhos menores de 6 anos) 
X_novo = pd.DataFrame({
    'husage': [40],
    'husunion': [0],
    'husearns' : [600],
    'huseduc': [13],
    'husblck': [1],
    'hushisp': [0],
    'hushrs': [40],
    'kidge6': [1],
    'earns': [355.5],
    'age': [38],
    'black': [0],
    'educ': [13],
    'hispanic': [1],
    'union': [0],
    'exper': [18],
    'kidlt6': [1]  
})

In [19]:
# calcular a predição
# cria um dataframe para guardar as medidas
df_pred = pd.DataFrame(columns=['y_novo', 'IC_inf', 'IC_sup'])

# insere no dataframe as predições calculadas de cada modelo convertida para o anti-log e inclui
# os intervalos de confiança para cada modelo

n = len(y_treino) # tamanho da amostra de treino
s = y_treino.std() # desvio padrao da amostra de treino
dam = s/np.sqrt(n) # distribuicao da amostragem da media

for name, info in trained_models.items():
    # calcula a predição
    y_hat_log = info['model'].predict(X_novo)[0]
    
    # calcula os Intervalos de confiança
    IC_inf = y_hat_log + (norm.ppf(0.025) * dam)
    IC_sup = y_hat_log - (norm.ppf(0.025) * dam)
    
    # converte e armazena
    df_pred.loc[name] = [round(np.exp(y_hat_log),2), round(np.exp(IC_inf), 2), round(np.exp(IC_sup), 2)]

print(df_pred)

            y_novo  IC_inf  IC_sup
Ridge         9.05    8.84    9.26
Lasso         8.91    8.71    9.11
ElasticNet    8.91    8.71    9.12
